# Tutorial 03 · CRISP-DM in practice: understand the data
**Data Mining · HTW Berlin**

In the lecture you saw the six CRISP-DM phases and how the semester follows them. Today we do the **first two phases for real**, on one small case:

| CRISP-DM phase | Today |
|---|---|
| 1 Business Understanding | what is the question, what is one row, what do we predict |
| 2 Data Understanding | what kind of column is each one, clean the values, compare categories |
| 3 Data Preparation | only a first look: what we found and when we treat it |
| 4 to 6 | later in the semester |

**How it works:** first an **Example**, then **Your turn** with almost the same code. Replace every `____` and run the cell. The answers are at the end of the notebook.

## 0 · Setup (just run, you do not need to read it)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)


def make_bookings(seed=4):
    """1,500 class bookings of a Berlin gym, April to June 2026 (made-up data)."""
    rng = np.random.default_rng(seed)
    n = 1500
    # members: plan and months as member stay the same for one person
    members = pd.DataFrame({"member_id": np.arange(100, 500)})
    members["plan"] = rng.choice(["Basic", "Plus", "Premium"], len(members), p=[0.5, 0.3, 0.2])
    members["months_member"] = np.clip(rng.gamma(2.0, 9, len(members)).round(), 1, 72).astype(int)
    members["habit"] = rng.normal(0, 0.6, len(members))
    members["past_no_shows"] = rng.poisson(np.exp(members["habit"]) * 1.2)

    df = pd.DataFrame({"member_id": rng.choice(members["member_id"], n)})
    df = df.merge(members, on="member_id", how="left")
    days = pd.date_range("2026-04-06", "2026-06-28")
    df["date"] = rng.choice(days, n)
    df["class_type"] = rng.choice(["Spinning", "Yoga", "HIIT", "Pilates"], n, p=[0.35, 0.25, 0.25, 0.15])
    df["class_time"] = rng.choice(["07:00", "09:00", "12:30", "18:00", "19:30"], n, p=[0.2, 0.15, 0.1, 0.3, 0.25])
    _ = np.where(rng.random(n) < 0.5, rng.uniform(0.5, 24, n), rng.uniform(24, 96, n))   # keeps the random stream
    # many people book shortly before the class, few book days before: an exponential distribution
    df["hours"] = np.round(np.clip(np.random.default_rng(seed + 100).exponential(24, n), 0.5, 168), 1)

    # who does not come
    weekday = df["date"].dt.dayofweek
    z = (-2.93 + 0.7 * (df["plan"] == "Basic") - 0.6 * (df["plan"] == "Premium")
         + 0.8 * (df["class_time"] == "07:00")
         + 0.9 * ((weekday == 4) & (df["class_time"].isin(["18:00", "19:30"])))
         + 0.9 * (df["hours"] > 24) + 0.8 * (df["months_member"] <= 3)
         + 0.35 * df["past_no_shows"] + 0.6 * df["habit"])
    df["no_show"] = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

    # check in and check out, only for people who came
    start = pd.to_datetime(df["date"].dt.strftime("%Y-%m-%d") + " " + df["class_time"])
    cin = start + pd.to_timedelta(rng.integers(-10, 6, n), unit="m")
    cout = cin + pd.to_timedelta(rng.integers(50, 80, n), unit="m")
    came = df.index[df["no_show"] == 0]
    forgot = rng.choice(came, 6, replace=False)                    # forgot to scan out: system closes at 23:59
    cout[forgot] = pd.to_datetime(df.loc[forgot, "date"].dt.strftime("%Y-%m-%d") + " 23:59")
    rest = np.setdiff1d(came, forgot)
    typo = rng.choice(rest, 3, replace=False)                      # typing error: out before in
    cout[typo] = cin[typo] - pd.to_timedelta(60, unit="m")
    fmt = "%Y-%m-%d %H:%M"
    df["check_in"] = np.where(df["no_show"] == 0, cin.dt.strftime(fmt), None)
    df["check_out"] = np.where(df["no_show"] == 0, cout.dt.strftime(fmt), None)

    # the messy table, as it comes out of the booking system
    out = pd.DataFrame({
        "booking_id": [f"B{10001 + i}" for i in range(n)],
        "member_id": df["member_id"],
        "plan": df["plan"].astype(object),
        "months_member": df["months_member"].astype(str) + " months",
        "class_type": df["class_type"].astype(object),
        "class_date": df["date"].dt.strftime("%Y-%m-%d"),
        "class_time": df["class_time"],
        "hours_booked_before": df["hours"].astype(str) + " h",
        "past_no_shows": df["past_no_shows"],
        "check_in": df["check_in"],
        "check_out": df["check_out"],
        "no_show": df["no_show"],
    })
    idx = rng.permutation(n)
    out.loc[idx[:80], "plan"] = "Unknown"
    out.loc[idx[80:110], "months_member"] = np.nan
    out.loc[idx[110:140], "class_type"] = out.loc[idx[110:140], "class_type"].str.lower() + " "
    out.loc[idx[140:160], "class_type"] = "SPIN"
    out.loc[idx[160:185], "plan"] = out.loc[idx[160:185], "plan"].str.upper()
    out.loc[idx[185:205], "plan"] = " " + out.loc[idx[185:205], "plan"].str.lower()
    out.loc[idx[205], "past_no_shows"] = 99
    dup = out.sample(40, random_state=1)
    out = pd.concat([out, dup]).sort_values("booking_id").reset_index(drop=True)
    return out


def plot_waterfall(steps, title="From raw to ready"):
    """steps = [(name, start value), (name, change), ...]. Draws a waterfall chart."""
    names, bottoms, heights, colors, texts = [], [], [], [], []
    total = 0
    for i, (name, value) in enumerate(steps):
        names.append(name)
        if i == 0:
            total = value
            bottoms.append(0); heights.append(value); colors.append("#4C72B0"); texts.append(f"{value:,}")
        elif value < 0:
            total += value
            bottoms.append(total); heights.append(-value); colors.append("#C44E52"); texts.append(f"{value:,}")
        else:
            bottoms.append(total); heights.append(max(value, 2)); colors.append("#AAAAAA")
            texts.append("0 (kept)" if value == 0 else f"+{value:,}")
            total += value
    names.append("ready"); bottoms.append(0); heights.append(total); colors.append("#55A868"); texts.append(f"{total:,}")
    low = max(0, int((total - (steps[0][1] - total) * 2) // 50 * 50))
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.bar(names, heights, bottom=bottoms, color=colors)
    for i in range(len(names)):
        ax.text(i, bottoms[i] + heights[i] + (steps[0][1] - low) * 0.02, texts[i], ha="center")
    ax.set_ylim(low, steps[0][1] + (steps[0][1] - low) * 0.15)
    ax.set_ylabel("rows")
    ax.set_title(title + f" (y axis starts at {low:,})")
    plt.xticks(rotation=20, ha="right")
    plt.show()


bookings = make_bookings()
print("Ready:", bookings.shape[0], "bookings,", bookings.shape[1], "columns")

---
# Part 1 · Business Understanding

**Kiez Gym** is a gym in Berlin with group classes: Spinning, Yoga, HIIT and Pilates. Members book a spot in the app. The problem: about one in four people who book **do not come**. The spot stays empty, while other members are on the waiting list.

| | |
|---|---|
| Business question | How can we fill the spots that stay empty? |
| Who uses the answer | the front desk team: they send a reminder or give the spot to the waiting list |
| Data mining question | Which bookings will be a **no-show**? |
| What we predict (the **target**) | the column `no_show`: 1 = did not come, 0 = came |
| One row is | one booking of one member for one class |

Before any model, we need to know the data. That is today.

**Example**

In [ ]:
# Step 1: the first rows
bookings.head()

In [ ]:
# Step 2: how many rows and columns?
bookings.shape

1,540 bookings and 12 columns. Look at the columns: some contain text where you expect numbers (`12.5 h`), some are empty (`NaN`). We will fix that in Part 3.

**Always start with a simple picture.** Before any number, look at the target: how many came, how many did not?

**Example**

In [ ]:
# Step 3: came or no-show, as a bar chart
counts = bookings["no_show"].value_counts()
counts.index = ["came", "no-show"]
counts.plot(kind="bar", rot=0, figsize=(4, 3), title="Came or not?")
plt.show()

You see it at once: most people come, but the no-show bar is not small. Now the exact numbers.

**Your turn · 1.1 · How many people do not come?**

<details><summary>Hint</summary>

```python
value_counts()
value_counts(normalize=True)
```

</details>

In [ ]:
# Step 1: count the 0s and the 1s in the target
bookings["no_show"].____()

*You should see:* 1,187 bookings with 0 (came) and 353 with 1 (did not come).

In [ ]:
# Step 2: the same as a share (normalize=True gives parts of 1)
bookings["no_show"].value_counts(normalize=____)

*You should see:* 0.77 and 0.23: **23% of the bookings are no-shows**.

This 23% is our **reference**. If the front desk calls 100 random bookings, about 23 are no-shows. A useful method must find **more** than that. This is how a business goal becomes a number you can test (Phase 1 output: the success criterion).

---
# Part 2 · What kind of column is it?

Every column has a **type**. The type decides what you can do with it later (which plot, which cleaning, how a model can use it).

| Type | Meaning | Example here | Why it matters later |
|---|---|---|---|
| **ID** | a name for a row or a person | `booking_id`, `member_id` | never a model input: the number means nothing |
| **numeric** | a quantity, you can calculate with it | `months_member`, `hours_booked_before`, `past_no_shows` | outliers, scaling |
| **categorical** | a group, no order | `class_type` | must become numbers for a model (encoding) |
| **ordinal** | a group **with an order** | `plan`: Basic < Plus < Premium | the order is information |
| **date / time** | a moment | `class_date`, `class_time`, `check_in`, `check_out` | we take parts from it: weekday, hour, duration |
| **target** | what we want to predict | `no_show` | never an input |

Python does not know these types. It only knows **numbers** (`int64`, `float64`) and **text** (`object`, or `str` in the newest pandas). You have to decide.

**Example**

In [ ]:
# Step 1: what Python thinks each column is
bookings.dtypes

In [ ]:
# Step 2: how many different values each column has
bookings.nunique()

- `months_member` and `hours_booked_before` are text (`object` or `str`) because of the `" months"` and `" h"`. Python cannot calculate with them yet.
- `class_type` has **9** different values, but the gym has only 4 classes. Something is written in different ways.
- Few different values usually means a **category**. Very many usually means a **number** or an **ID**.

**Example**

In [ ]:
# Step 3: describe() calculates everything that is a number
bookings.describe().round(1)

The **mean member_id is 297.6**. Python calculated it because the column contains numbers, but it means nothing: member 297 is not an "average member". `member_id` is an ID, not a quantity.

Also look at `past_no_shows`: the 75% quartile is 2, the max is **99**. 99 earlier no-shows for one member is suspicious. Values like this are **outliers**, a topic of its own.

**Your turn · 2.1 · Give each column its type**

<details><summary>Hint</summary>

```python
"ID"  "numeric"  "categorical"  "ordinal"  "date"  "target"
```

</details>

In [ ]:
# Step 1: write the type of each column: "ID", "numeric", "categorical", "ordinal", "date", "target"
column_types = {
    "booking_id": "ID",
    "member_id": ____,
    "plan": ____,
    "months_member": "numeric",
    "class_type": ____,
    "class_date": "date",
    "past_no_shows": ____,
    "no_show": ____,
}
column_types

*You should see:* the dictionary printed with a type for each column. Compare with the table at the top of Part 2.

---
# Part 3 · Clean the values
Five problems that you will meet in almost every real dataset.

## 3.1 · Duplicates
The booking system sometimes saves the same booking twice.

**Example**

In [ ]:
# Step 1: how many rows are exact copies of another row?
print(bookings.duplicated().sum())

In [ ]:
# Step 2: remove them and check the new size
bookings = bookings.drop_duplicates()
bookings.shape

40 copies removed: 1,500 bookings are left. If we kept them, these 40 bookings would count twice in every result.

## 3.2 · Categories written in different ways
For Python, `"Spinning"`, `"spinning "` and `"SPIN"` are three different classes.

**Example**

In [ ]:
# Step 1: look at all the values
bookings["class_type"].value_counts()

In [ ]:
# Step 2: remove spaces at the start and end, and write everything in small letters
bookings["class_type"] = bookings["class_type"].str.strip().str.lower()
bookings["class_type"].value_counts()

In [ ]:
# Step 3: "spin" is a short name for "spinning"
bookings["class_type"] = bookings["class_type"].replace("spin", "spinning")
bookings["class_type"].value_counts()

From 9 values to the 4 real classes. `.str` means: do this to the **text** in every row.

**Your turn · 3.2 · Clean the plan**

<details><summary>Hint</summary>

```python
.str.strip().str.lower()
```

</details>

In [ ]:
# Step 1: look at all the values
bookings["plan"].value_counts()

*You should see:* 10 values: `Basic`, `BASIC`, `" basic"`, and so on.

In [ ]:
# Step 2: remove spaces and write in small letters
bookings["plan"] = bookings["plan"].str.____().str.____()
bookings["plan"].value_counts()

*You should see:* 4 values: basic 620, plus 476, premium 324, unknown 80.

**What about `unknown`?** 80 members have no plan in the system. It is tempting to delete them or to guess a plan. But "we do not know the plan" can be information too (maybe these are trial members). For now we **keep `unknown` as its own group** and see in Part 4 if it behaves differently.

## 3.3 · Numbers saved as text
`"11.0 h"` is text. We remove the `" h"` and tell Python it is a number.

**Example**

In [ ]:
# Step 1: what it looks like now
bookings["hours_booked_before"].head()

In [ ]:
# Step 2: remove " h"
bookings["hours_booked_before"] = bookings["hours_booked_before"].str.replace(" h", "")
bookings["hours_booked_before"].head()

In [ ]:
# Step 3: turn the text into numbers
bookings["hours_booked_before"] = pd.to_numeric(bookings["hours_booked_before"])
bookings["hours_booked_before"].describe().round(1)

In [ ]:
# Step 4: now it is a number, so we can draw it
bookings["hours_booked_before"].plot(kind="hist", bins=30, figsize=(6, 3), title="Hours booked before the class")
plt.show()

It still looks the same in Step 2, but look at the last line: the dtype is still text (`object` or `str`). After Step 3, `describe()` works: people book between 0.5 and 168 hours (one week) before the class. Step 4 was impossible with text.

**How to read the histogram:** many people book a few hours before the class, and the more hours, the fewer people. The bars fall fast on the left and then end in a long, flat **tail** to the right. This shape is called **right-skewed** (or a long tail). You will see it again and again: income, prices, waiting times.

One sign of a right-skewed column: the **mean** (24 h) is bigger than the **median** (17 h), because the few very early bookings pull the mean to the right.

**Your turn · 3.3 · Months as member**

<details><summary>Hint</summary>

```python
.str.replace(" months", "")
pd.to_numeric(...)
```

</details>

In [ ]:
# Step 1: what it looks like now
bookings["months_member"].head()

*You should see:* texts like `"8 months"`.

In [ ]:
# Step 2: remove " months"
bookings["months_member"] = bookings["months_member"].str.replace(____, "")
bookings["months_member"].head()

*You should see:* only the numbers, but the dtype is still text.

In [ ]:
# Step 3: turn the text into numbers
bookings["months_member"] = pd.____(bookings["months_member"])
bookings["months_member"].describe().round(1)

*You should see:* count 1470 (not 1500!), median 13 months, max 67.

## 3.4 · Empty values
`count` was 1,470, not 1,500. Some values are missing (`NaN`).

**Example**

In [ ]:
# Step 1: how many empty values in each column?
bookings.isna().sum()

- `months_member`: **30** empty. A small number. How to fill them (imputation) is a topic later in the semester. Today we only **write it down**.
- `check_in` and `check_out`: **347** empty. That is a lot. Keep this number in mind for Part 5.

**Your turn · 3.4 · Count one column**

<details><summary>Hint</summary>

```python
.isna().sum()
```

</details>

In [ ]:
# How many empty values in months_member?
print(bookings["months_member"].____().____())

*You should see:* 30.

## 3.5 · Dates and times
`"2026-06-04"` is text. `pd.to_datetime` turns it into a date, and then we can take parts of it, like the weekday.

**Example**

In [ ]:
# Step 1: text to date
bookings["class_date"] = pd.to_datetime(bookings["class_date"])
bookings["class_date"].head()

In [ ]:
# Step 2: a new column with the weekday
bookings["weekday"] = bookings["class_date"].dt.day_name()
bookings[["class_date", "weekday"]].head()

`.dt` works like `.str`, but for dates: `.dt.day_name()`, `.dt.month`, `.dt.hour`.

Now the check-in and check-out. A class takes about 60 minutes. If we subtract the two times, we get how long the person stayed.

**Your turn · 3.5 · How long did people stay?**

<details><summary>Hint</summary>

```python
pd.to_datetime(...)
(bookings["minutes"] < 0) | (bookings["minutes"] > 180)
```

</details>

In [ ]:
# Step 1: text to date and time
bookings["check_in"] = pd.____(bookings["check_in"])
bookings["check_out"] = pd.____(bookings["check_out"])
bookings[["check_in", "check_out"]].head()

*You should see:* the same values, but now as dates (`NaT` = empty date).

In [ ]:
# Step 2: out minus in, in minutes
bookings["minutes"] = (bookings["check_out"] - bookings["check_in"]).dt.total_seconds() / 60
bookings["minutes"].describe().round(1)

*You should see:* median 64 minutes, but min **-60** and max **1023** (17 hours).

In [ ]:
# Step 3: a picture of the minutes
bookings["minutes"].plot(kind="box", vert=False, figsize=(7, 2), title="Minutes in the gym")
plt.show()

*You should see:* a thin box around 60 minutes and a few dots far away: on the left below 0, on the right up to 1023.

In [ ]:
# Step 4: the impossible visits: less than 0 minutes or more than 3 hours (180 minutes)
bookings[(bookings["minutes"] < ____) | (bookings["minutes"] > ____)]

*You should see:* 9 rows: 3 with -60 (check-out before check-in, a typing error) and 6 with a check-out at **23:59** (the person forgot to scan out, the system closed the visit at midnight).

A **domain rule** (a class cannot take a negative time or 17 hours) finds errors, and the values (23:59) tell you **why** they happened.

---
# Part 4 · Compare the groups with the target
Now the clean columns pay off. Which groups do not come more often?

**The trick:** `no_show` is 0 or 1. The **mean** of 0s and 1s is the share of 1s. `mean() = 0.30` means 30% no-shows.

**Example**

In [ ]:
# Step 1: first a picture: in each plan, the share of came and no-show
table = pd.crosstab(bookings["plan"], bookings["no_show"], normalize="index")
table.columns = ["came", "no-show"]
table.plot(kind="barh", stacked=True, figsize=(6, 2.5), title="Came or not, per plan")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
# Step 2: the numbers: no-show rate per plan
bookings.groupby("plan")["no_show"].mean().round(2)

In [ ]:
# Step 3: another picture of the same numbers: only the no-show rate, with the 23% reference as a line
rate = bookings.groupby("plan")["no_show"].mean().sort_values()
rate.plot(kind="barh", figsize=(6, 2.5), title="No-show rate per plan")
plt.axvline(0.23, color="grey", linestyle="--")
plt.show()

**How to read Step 1:** every bar is one plan and adds up to 100%. The orange part is the share of no-shows: the longer it is, the more often this group does not come. **Step 3** shows only the no-show rate, with the grey line at 23% (all bookings). Both pictures tell the same story; choose the one your audience reads faster.

Basic members do not come in **30%** of their bookings, Premium only **13%**. Premium members pay more, maybe they plan better. `unknown` (22%) is close to the average, so the empty plan does not look like a special group.

**Your turn · 4.1 · Weekday**

<details><summary>Hint</summary>

```python
bookings["weekday"]
groupby("weekday")
```

</details>

In [ ]:
# Step 1: first a picture: in each weekday, the share of came and no-show
table = pd.crosstab(bookings[____], bookings["no_show"], normalize="index")
table.columns = ["came", "no-show"]
table.plot(kind="barh", stacked=True, figsize=(6, 3), title="Came or not, per weekday")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

*You should see:* seven bars; the orange part on **Friday** is clearly longer.

In [ ]:
# Step 2: no-show rate per weekday, sorted
bookings.groupby(____)["no_show"].mean().sort_values().round(2)

*You should see:* most days between 0.16 and 0.27, but **Friday 0.35**.

## Weekend or not?
Seven days are seven groups. A simpler column has only two: **weekend** or **not**. A comparison like `>= 5` gives `True` or `False` for every row.

**Example**

In [ ]:
# Step 1: dayofweek gives a number: Monday = 0, Tuesday = 1, ... Saturday = 5, Sunday = 6
bookings["day_number"] = bookings["class_date"].dt.dayofweek
bookings[["weekday", "day_number"]].head()

In [ ]:
# Step 2: weekend = day 5 or 6. The comparison gives True or False
bookings["is_weekend"] = bookings["day_number"] >= 5
bookings[["weekday", "day_number", "is_weekend"]].head()

In [ ]:
# Step 3: first a picture: in each group, the share of came and no-show
table = pd.crosstab(bookings["is_weekend"], bookings["no_show"], normalize="index")
table.columns = ["came", "no-show"]
table.plot(kind="barh", stacked=True, figsize=(6, 2), title="Came or not: is_weekend")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
# Step 4: no-show rate: False = Monday to Friday, True = weekend
bookings.groupby("is_weekend")["no_show"].mean().round(2)

Almost **no difference** (0.24 against 0.22). The weekend column **hides** the one day that matters: Friday. A simpler column is good only if it keeps the pattern you saw in the picture. Here a better simple column is *Friday or not*.

**Your turn · 4.2 · Friday or not**

<details><summary>Hint</summary>

```python
"Friday"
groupby("is_friday")
```

</details>

In [ ]:
# Step 1: True if the class is on a Friday
bookings["is_friday"] = bookings["weekday"] == ____
bookings[["weekday", "is_friday"]].head()

*You should see:* the weekday and a new column with True only on Fridays.

In [ ]:
# Step 2: no-show rate: False = other days, True = Friday
bookings.groupby(____)["no_show"].mean().round(2)

*You should see:* False 0.21, True **0.35**. One True/False column keeps the whole pattern.

**Your turn · 4.3 · Class type**

<details><summary>Hint</summary>

```python
bookings["class_type"]
groupby("class_type")
```

</details>

In [ ]:
# Step 1: first a picture: in each class type, the share of came and no-show
table = pd.crosstab(bookings[____], bookings["no_show"], normalize="index")
table.columns = ["came", "no-show"]
table.plot(kind="barh", stacked=True, figsize=(6, 2.5), title="Came or not, per class type")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

*You should see:* four bars with almost the same orange part.

In [ ]:
# Step 2: no-show rate per class type
bookings.groupby(____)["no_show"].mean().round(2)

*You should see:* all four classes between 0.22 and 0.24.

The class type makes **no difference**. That is also a result: not every column helps to predict. Finding this now saves work later.

## Numbers into groups
**Two groups:** a comparison is enough, the same as `is_weekend`. People who booked more than 24 hours before get `True`.

**Example**

In [ ]:
# Step 1: True if the booking was made more than 24 hours before the class
bookings["booked_early"] = bookings["hours_booked_before"] > 24
bookings[["hours_booked_before", "booked_early"]].head()

In [ ]:
# Step 2: first a picture: in each group, the share of came and no-show
table = pd.crosstab(bookings["booked_early"], bookings["no_show"], normalize="index")
table.columns = ["came", "no-show"]
table.plot(kind="barh", stacked=True, figsize=(6, 2), title="Came or not: booked_early")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
# Step 3: no-show rate: False = up to 24 h before, True = more than 24 h before
bookings.groupby("booked_early")["no_show"].mean().round(2)

Who books more than a day before does not come almost **twice as often** (0.32 against 0.18). Plans change in a day.

**Three or more groups:** `pd.cut`. Think of a ruler: you **cut** it at some borders and give each piece a name. First a small example with six numbers, so you can check every row by eye.

**Example**

In [ ]:
# Step 1: six made-up values of months as member
months = pd.Series([1, 3, 4, 12, 13, 40])
months

In [ ]:
# Step 2: the borders and the names
# borders 0 | 3 | 12 | 100 make three pieces: 0 to 3, 3 to 12, 12 to 100
borders = [0, 3, 12, 100]
names = ["new", "regular", "loyal"]     # one name per piece: 4 borders, 3 names
print(borders)
print(names)

In [ ]:
# Step 3: cut the six values into the pieces
pd.DataFrame({"months": months, "group": pd.cut(months, bins=borders, labels=names)})

A border belongs to the piece on its **left**: 3 is still *new*, 4 is *regular*; 12 is still *regular*, 13 is *loyal*.

Now the same on the real column.

**Your turn · 4.4 · New members**

<details><summary>Hint</summary>

```python
bins=borders, labels=names
bookings["member_group"]
groupby("member_group", observed=True)
```

</details>

In [ ]:
# Step 1: the same borders and names as in the small example, now on the real column
bookings["member_group"] = pd.cut(bookings["months_member"], bins=____, labels=____)
bookings[["months_member", "member_group"]].head()

*You should see:* a new column with new, regular or loyal.

In [ ]:
# Step 2: first a picture: in each group, the share of came and no-show
table = pd.crosstab(bookings[____], bookings["no_show"], normalize="index")
table.columns = ["came", "no-show"]
table.plot(kind="barh", stacked=True, figsize=(6, 2.5), title="Came or not, per group")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

*You should see:* three bars; the orange part is longest for **new** members.

In [ ]:
# Step 3: no-show rate per group
bookings.groupby(____, observed=True)["no_show"].mean().round(2)

*You should see:* new 0.37, regular 0.24, loyal 0.20.

**What we learned in Part 4**

| Column | Difference in no-show rate | Useful to predict? |
|---|---|---|
| plan | 13% to 30% | probably yes |
| weekday | Friday 35%, others 16% to 27% | probably yes |
| weekend or not | 22% vs 24% | no: it hides Friday |
| Friday or not | 35% vs 21% | yes, and simpler than 7 days |
| booked before | 18% vs 32% | probably yes |
| months as member | new members 37% | probably yes |
| class type | all around 23% | probably not |

---
# Part 5 · From raw to ready
In Part 3 we changed the table step by step. In a real project you do it in a fixed order:

1. **Mark** the problems first, without changing anything.
2. **Count** them, so you can report them.
3. Only then **remove or fix** them.

## 5.1 · Mark the problems with flag columns
A **flag** is a new column with `True` (this row has the problem) or `False` (it does not). The original values stay untouched, so you can count the problems, show them to the person who owns the data, and change your mind later.

We start again from the **raw** table. Duplicates are copies, not real bookings, so we remove them right away. Then we mark the rest.

**Example**

In [ ]:
# Step 1: the raw table again, without the duplicates
raw = make_bookings()
clean = raw.drop_duplicates().copy()
print("raw:", len(raw), "rows  ->  without duplicates:", len(clean), "rows")

In [ ]:
# Step 2: one flag column per problem (True = problem)
clean["flag_impossible"] = clean["past_no_shows"] == 99
clean["flag_empty_months"] = clean["months_member"].isna()
clean[["booking_id", "past_no_shows", "flag_impossible", "months_member", "flag_empty_months"]].head()

In [ ]:
# Step 3: count the flags: True counts as 1, False as 0
clean[["flag_impossible", "flag_empty_months"]].sum()

In [ ]:
# Step 4: look at the flagged rows before you delete them
clean[clean["flag_impossible"] | clean["flag_empty_months"]].head()

1 booking with an impossible value and 30 with empty months. Step 4 is the moment to ask: *is this really an error?* Here the answer is yes.

**Your turn · 5.1 · Flag the unknown plans**

<details><summary>Hint</summary>

```python
"unknown"
.sum()
```

</details>

In [ ]:
# Step 1: a flag for the plans that are "unknown" (after cleaning the text)
clean["flag_plan_unknown"] = clean["plan"].str.strip().str.lower() == ____
clean[["plan", "flag_plan_unknown"]].head()

*You should see:* the plan and a new column with True or False.

In [ ]:
# Step 2: how many?
print(clean["flag_plan_unknown"].____())

*You should see:* 80.

We **keep** the unknown plans (they are a group of their own), but the flag stays in the table. Later you can always check: does the model work worse for these 80 bookings?

**Example**

In [ ]:
# Step 5: remove the flagged rows we decided to remove. ~ means "not"
ready = clean[~clean["flag_impossible"] & ~clean["flag_empty_months"]].copy()
print(len(ready), "rows ready")

In [ ]:
# Step 6: the cleaning recipe from Part 3, all in one cell
ready["plan"] = ready["plan"].str.strip().str.lower()
ready["class_type"] = ready["class_type"].str.strip().str.lower().replace("spin", "spinning")
ready["months_member"] = pd.to_numeric(ready["months_member"].str.replace(" months", ""))
ready["hours_booked_before"] = pd.to_numeric(ready["hours_booked_before"].str.replace(" h", ""))
ready["weekday"] = pd.to_datetime(ready["class_date"]).dt.day_name()
ready.head()

1,540 → 1,500 → **1,469 rows ready**. We **removed** the 30 rows with empty months to keep it simple today. Later in the semester you will learn to **fill** them (imputation) and get these rows back.

Step 6 is your **cleaning recipe**: anyone can run it again on new data and get the same table.

## 5.2 · Show what happened to the rows
**Picture 1: a waterfall.** It starts with all rows (blue), every red bar is a step that removes rows, grey bars are problems we **kept**, and the green bar is what is left. The function `plot_waterfall` is in the setup: you only give it the steps.

**Your turn · 5.2 · Your waterfall**

<details><summary>Hint</summary>

```python
-1
-30
```

</details>

In [ ]:
# Each step: (name, change in rows). Use the counts from 5.1
steps = [
    ("raw", 1540),
    ("duplicates", -40),
    ("past_no_shows = 99", ____),
    ("empty months", ____),
    ("plan unknown (kept)", 0),
]
plot_waterfall(steps)

*You should see:* a blue bar at 1,540, red steps of -40, -1 and -30, a grey step of 0, and a green bar at **1,469**. The y axis starts at 1,300, otherwise the small steps would be invisible.

**Picture 2 and 3: the same story, simpler.** Use the one that fits your audience: a manager wants Picture 2, a data team wants the waterfall or Picture 3.

**Example**

In [ ]:
# Picture 2: before and after, two bars
pd.Series({"raw": len(raw), "ready": len(ready)}).plot(
    kind="bar", rot=0, color=["#4C72B0", "#55A868"], figsize=(4, 3), title="Rows before and after")
plt.show()

In [ ]:
# Picture 3: why rows were removed, biggest reason on top
removed = pd.Series({"duplicates": 40, "past_no_shows = 99": 1, "empty months": 30}).sort_values()
removed.plot(kind="barh", color="#C44E52", figsize=(6, 2.5), title="Rows removed, by reason")
plt.show()

95% of the data survived. The biggest loss is the duplicates, and they were never real bookings.

---
# Wrap up · The data quality report
In CRISP-DM, Phase 2 ends with a **data quality report**: what is wrong, the evidence, and what we plan to do. Here is ours, made with the numbers we found today.

In [ ]:
report = pd.DataFrame({
    "issue": ["duplicate rows", "category spelling", "numbers saved as text", "empty months_member",
              "plan unknown", "impossible visit times", "past_no_shows = 99", "check_in / check_out"],
    "evidence": ["40 rows", "9 class names for 4 classes", "2 columns", "30 rows",
                 "80 rows", "9 rows (-60 min, 23:59)", "1 row", "empty for every no-show"],
    "response": ["removed (3.1)", "cleaned (3.2)", "converted (3.3)", "fill later (imputation)",
                 "keep as own group, flagged", "only matters for duration, not an input", "check later (outliers)", "not for prediction: only known after the class"],
})
report

## Where this goes in the semester
| What we found today | CRISP-DM phase | When we work on it |
|---|---|---|
| `past_no_shows = 99` | 3 Data Preparation | outliers |
| 30 empty months | 3 Data Preparation | missing values and imputation |
| plan, class type, weekday are text | 3 Data Preparation | encoding: categories into numbers |
| hours from 0.5 to 168, months from 1 to 67 | 3 Data Preparation | scaling |
| class type does not help, plan does | 3 Data Preparation | feature selection |
| the columns known before the class, target `no_show` | 4 Modeling | classification models |
| reference: 23% no-shows | 5 Evaluation | does the model find more than 23%? |
| a reminder list for the front desk | 6 Deployment | end of the semester |

**Today you did what takes most of the time in a real project: understanding the data.** Every model later in the semester starts from a table like the one you cleaned today.

---
## Answers
Try first, then compare. Click to open.

<details><summary>1.1 · How many people do not come?</summary>

```python
# Step 1: count the 0s and the 1s in the target
bookings["no_show"].value_counts()

# Step 2: the same as a share (normalize=True gives parts of 1)
bookings["no_show"].value_counts(normalize=True)
```

</details>

<details><summary>2.1 · Give each column its type</summary>

```python
# Step 1: write the type of each column: "ID", "numeric", "categorical", "ordinal", "date", "target"
column_types = {
    "booking_id": "ID",
    "member_id": "ID",
    "plan": "ordinal",
    "months_member": "numeric",
    "class_type": "categorical",
    "class_date": "date",
    "past_no_shows": "numeric",
    "no_show": "target",
}
column_types
```

</details>

<details><summary>3.2 · Clean the plan</summary>

```python
# Step 1: look at all the values
bookings["plan"].value_counts()

# Step 2: remove spaces and write in small letters
bookings["plan"] = bookings["plan"].str.strip().str.lower()
bookings["plan"].value_counts()
```

</details>

<details><summary>3.3 · Months as member</summary>

```python
# Step 1: what it looks like now
bookings["months_member"].head()

# Step 2: remove " months"
bookings["months_member"] = bookings["months_member"].str.replace(" months", "")
bookings["months_member"].head()

# Step 3: turn the text into numbers
bookings["months_member"] = pd.to_numeric(bookings["months_member"])
bookings["months_member"].describe().round(1)
```

</details>

<details><summary>3.4 · Count one column</summary>

```python
# How many empty values in months_member?
print(bookings["months_member"].isna().sum())
```

</details>

<details><summary>3.5 · How long did people stay?</summary>

```python
# Step 1: text to date and time
bookings["check_in"] = pd.to_datetime(bookings["check_in"])
bookings["check_out"] = pd.to_datetime(bookings["check_out"])
bookings[["check_in", "check_out"]].head()

# Step 2: out minus in, in minutes
bookings["minutes"] = (bookings["check_out"] - bookings["check_in"]).dt.total_seconds() / 60
bookings["minutes"].describe().round(1)

# Step 3: a picture of the minutes
bookings["minutes"].plot(kind="box", vert=False, figsize=(7, 2), title="Minutes in the gym")
plt.show()

# Step 4: the impossible visits: less than 0 minutes or more than 3 hours (180 minutes)
bookings[(bookings["minutes"] < 0) | (bookings["minutes"] > 180)]
```

</details>

<details><summary>4.1 · Weekday</summary>

```python
# Step 1: first a picture: in each weekday, the share of came and no-show
table = pd.crosstab(bookings["weekday"], bookings["no_show"], normalize="index")
table.columns = ["came", "no-show"]
table.plot(kind="barh", stacked=True, figsize=(6, 3), title="Came or not, per weekday")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

# Step 2: no-show rate per weekday, sorted
bookings.groupby("weekday")["no_show"].mean().sort_values().round(2)
```

</details>

<details><summary>4.2 · Friday or not</summary>

```python
# Step 1: True if the class is on a Friday
bookings["is_friday"] = bookings["weekday"] == "Friday"
bookings[["weekday", "is_friday"]].head()

# Step 2: no-show rate: False = other days, True = Friday
bookings.groupby("is_friday")["no_show"].mean().round(2)
```

</details>

<details><summary>4.3 · Class type</summary>

```python
# Step 1: first a picture: in each class type, the share of came and no-show
table = pd.crosstab(bookings["class_type"], bookings["no_show"], normalize="index")
table.columns = ["came", "no-show"]
table.plot(kind="barh", stacked=True, figsize=(6, 2.5), title="Came or not, per class type")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

# Step 2: no-show rate per class type
bookings.groupby("class_type")["no_show"].mean().round(2)
```

</details>

<details><summary>4.4 · New members</summary>

```python
# Step 1: the same borders and names as in the small example, now on the real column
bookings["member_group"] = pd.cut(bookings["months_member"], bins=borders, labels=names)
bookings[["months_member", "member_group"]].head()

# Step 2: first a picture: in each group, the share of came and no-show
table = pd.crosstab(bookings["member_group"], bookings["no_show"], normalize="index")
table.columns = ["came", "no-show"]
table.plot(kind="barh", stacked=True, figsize=(6, 2.5), title="Came or not, per group")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

# Step 3: no-show rate per group
bookings.groupby("member_group", observed=True)["no_show"].mean().round(2)
```

</details>

<details><summary>5.1 · Flag the unknown plans</summary>

```python
# Step 1: a flag for the plans that are "unknown" (after cleaning the text)
clean["flag_plan_unknown"] = clean["plan"].str.strip().str.lower() == "unknown"
clean[["plan", "flag_plan_unknown"]].head()

# Step 2: how many?
print(clean["flag_plan_unknown"].sum())
```

</details>

<details><summary>5.2 · Your waterfall</summary>

```python
# Each step: (name, change in rows). Use the counts from 5.1
steps = [
    ("raw", 1540),
    ("duplicates", -40),
    ("past_no_shows = 99", -1),
    ("empty months", -30),
    ("plan unknown (kept)", 0),
]
plot_waterfall(steps)
```

</details>